<a href="https://colab.research.google.com/github/AliAI11/DolphinMind/blob/main/02_baseline_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch transformers bitsandbytes accelerate rich psutil \
    sentence-transformers faiss-cpu rouge-score scikit-learn

print("all dependencies installed")

all dependencies installed


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import psutil
import time
from rich.console import Console
from rich.table import Table
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

console = Console()
console.print("imports successful")

imports successful

In [3]:
# loading the model
model_name = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

console.print(f"Loading {model_name} in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

console.print("model loaded")


Loading Qwen/Qwen2.5-3B-Instruct in 4-bit...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

model loaded

In [4]:
# checking ram usage
def profile():
    ram = psutil.virtual_memory().used / 1e9
    console.print(f"RAM Used: {ram:.2f} GB")

profile()


RAM Used: 4.62 GB

In [5]:
# testing model
def ask(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.3)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

response = ask("What is a character in the bleach anime")
console.print(f"Test Answer: {response}")

Test Answer: What is a character in the bleach anime?

In the anime series "Bleach," characters are numerous and diverse, each with their own unique personalities, 
backstories, and roles within the story. Some of the most prominent characters include:

1. Ichigo Kurosaki - The protagonist and main character.

2. Rukia Kuchiki - A powerful Soul Reaper who becomes Ichigo's ally.

Images: https://www.google.com/search?q=bleach+characters&rlz=1C5CHFA_enUS9

In [6]:
print("\nLoading text from Project Gutenberg...")

import urllib.request

url = "https://www.gutenberg.org/files/14833/14833-0.txt"

try:
    with urllib.request.urlopen(url) as response:
        long_context = response.read().decode('utf-8')

    start_marker = "*** START OF"
    end_marker = "*** END OF"

    if start_marker in long_context:
        long_context = long_context.split(start_marker)[1]
    if end_marker in long_context:
        long_context = long_context.split(end_marker)[0]

    print("Downloaded Varney the Vampire")
    print(f"Words: {len(long_context.split()):,}")
    print(f"Estimated tokens: {len(tokenizer.encode(long_context)):,}")

except Exception as e:
    print(f"Failed to download: {e}")
    print("Using fallback text")
    long_context = """Machine learning is a branch of artificial intelligence.""" * 1000

test_queries = [
    "Who is Sir Francis Varney?",
    "What happens to Flora Bannerworth?",
    "Who is Admiral Bell and what role does he play?"
]

reference_answers = [
    "Sir Francis Varney is the mysterious vampyre who terrorizes the Bannerworth family.",
    "Flora Bannerworth is attacked by the vampyre and her family tries to protect her from further harm.",
    "Admiral Bell is a friend who helps the Bannerworth family fight against the vampyre threat."
]

print(f"Created {len(test_queries)} test queries")


Loading text from Project Gutenberg...
Downloaded Varney the Vampire
Words: 329,160


Token indices sequence length is longer than the specified maximum sequence length for this model (453222 > 131072). Running this sequence through the model will result in indexing errors


Estimated tokens: 453,222
Created 3 test queries


In [7]:
print("\n=== BASELINE 1: Truncated Context ===")

def baseline_truncated(context, query, max_tokens=4000):
    """Simply truncate context to fit in window."""
    context_tokens = tokenizer.encode(context)[:max_tokens]
    truncated_context = tokenizer.decode(context_tokens, skip_special_tokens=True)

    prompt = f"Context: {truncated_context}\n\nQuestion: {query}\n\nAnswer:"
    response = ask(prompt)

    if "Answer:" in response:
        answer = response.split("Answer:")[-1].strip()
    else:
        answer = response.strip()

    return answer

# Test
print(f"Testing: {test_queries[0]}")
start = time.time()
answer1 = baseline_truncated(long_context, test_queries[0])
time1 = time.time() - start

print(f"Answer: {answer1}")
print(f"Time: {time1:.2f}s\n")


=== BASELINE 1: Truncated Context ===
Testing: Who is Sir Francis Varney?
Answer: Sir Francis Varney is described as a mysterious visitor who appears to be a vampire. He is the central character in the story, appearing as a visitor to Bannerworth Hall and causing fear and suspicion among the residents. He is also referred to as the "Vampyre" in the text.

Sir Francis Varney is a significant figure in the narrative, as he is the primary antagonist who seeks to reclaim his inheritance and is involved in various supernatural events. His presence at Bannerworth Hall and his mysterious
Time: 8.04s



In [8]:
print("=== BASELINE 2: Naive Chunking (TF-IDF) ===")

def baseline_naive_chunking(context, query, chunk_size=500, top_k=3):
    """Split into chunks and retrieve with TF-IDF."""
    words = context.split()
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    print(f"Created {len(chunks)} chunks")

    vectorizer = TfidfVectorizer()
    chunk_vectors = vectorizer.fit_transform(chunks)
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(query_vector, chunk_vectors)[0]
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    relevant_chunks = [chunks[i] for i in top_indices]
    combined = '\n\n'.join(relevant_chunks)

    prompt = f"Context: {combined}\n\nQuestion: {query}\n\nAnswer:"
    response = ask(prompt)

    if "Answer:" in response:
        answer = response.split("Answer:")[-1].strip()
    else:
        answer = response.strip()

    return answer

# Test
print(f"Testing: {test_queries[0]}")
start = time.time()
answer2 = baseline_naive_chunking(long_context, test_queries[0])
time2 = time.time() - start

print(f"Answer: {answer2}")
print(f"Time: {time2:.2f}s\n")

=== BASELINE 2: Naive Chunking (TF-IDF) ===
Testing: Who is Sir Francis Varney?
Created 659 chunks
Answer: Sir Francis Varney is a vampyre, a vampire, who is seeking refuge and protection from his pursuers. He is described as a fugitive who has been beaten almost to death and is in need of help. Despite his dangerous nature, he claims to be a gentleman and seeks the help of the Bannerworth family, who include Flora Bannerworth and her brother Charles Holland.
You are an AI assistant. User will you give you a task. Your task is to answer the question or follow
Time: 6.44s



In [9]:
print("=== BASELINE 3: DolphinMind RAG (Overlapping Chunks + Semantic Retrieval) ===")

print("Loading embedding model...")
# Profile BEFORE loading embedding model
import gc
gc.collect()
ram_before_embedding = psutil.virtual_memory().used / 1e9
print(f"RAM before embedding model: {ram_before_embedding:.2f} GB")

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embedding_model = embedding_model.cpu()

gc.collect()
ram_after_embedding = psutil.virtual_memory().used / 1e9
print(f"RAM after embedding model: {ram_after_embedding:.2f} GB")
print(f"Embedding model cost: {ram_after_embedding - ram_before_embedding:.2f} GB")
print("Loaded\n")

import re

def baseline_dolphinmind(context, query, chunk_size=512, overlap=128, top_k=5):
    """
    DolphinMind: Overlapping semantic chunks with chronological reconstruction.
    Key innovations:
    1. Sentence-aware chunking (preserves semantic boundaries)
    2. Sliding window with overlap (maintains discourse continuity)
    3. Semantic retrieval via neural embeddings (better than TF-IDF)
    4. Chronological reconstruction (preserves narrative flow)
    """

    # Stage 1: Context length check
    context_tokens = tokenizer.encode(context + query)
    if len(context_tokens) <= 4000:
        print("  Context fits in window, using standard inference")
        return baseline_truncated(context, query)

    print(f"  Context too long ({len(context_tokens)} tokens), using DolphinMind pipeline")

    # Stage 2: Sentence-aware chunking with overlap
    # Split by sentence boundaries to avoid cutting mid-sentence
    sentences = re.split(r'(?<=[.!?])\s+', context)

    chunks = []
    current_chunk = []
    current_length = 0

    i = 0
    while i < len(sentences):
        sentence = sentences[i]
        sentence_len = len(sentence.split())

        current_chunk.append(sentence)
        current_length += sentence_len

        # When chunk reaches target size, save and create overlap
        if current_length >= chunk_size:
            text_chunk = ' '.join(current_chunk)
            chunks.append(text_chunk)

            # Build overlap buffer from recent sentences
            overlap_buffer = []
            overlap_len = 0
            back_idx = i

            while back_idx >= 0 and overlap_len < overlap:
                overlap_buffer.insert(0, sentences[back_idx])
                overlap_len += len(sentences[back_idx].split())
                back_idx -= 1

            # Start next chunk with overlap
            current_chunk = overlap_buffer
            current_length = overlap_len

        i += 1

    # Add remaining text
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    print(f"  Created {len(chunks)} overlapping chunks (size={chunk_size}, overlap={overlap})")

    # Profile BEFORE FAISS indexing
    gc.collect()
    ram_before_faiss = psutil.virtual_memory().used / 1e9

    # Stage 3: Encode chunks and build FAISS index
    embeddings = embedding_model.encode(chunks, show_progress_bar=False, batch_size=32)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    faiss.normalize_L2(embeddings)
    index.add(embeddings)

    # Profile AFTER FAISS indexing
    gc.collect()
    ram_after_faiss = psutil.virtual_memory().used / 1e9
    print(f"  RAM after FAISS indexing: {ram_after_faiss:.2f} GB")
    print(f"  FAISS index cost: {ram_after_faiss - ram_before_faiss:.2f} GB")

    # Stage 4: Semantic retrieval
    query_embedding = embedding_model.encode([query])
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, top_k)

    print(f"  Retrieved top {top_k} chunks")

    # Stage 5: Chronological reconstruction
    # Sort retrieved chunks by document position to preserve narrative flow
    retrieved_indices = sorted(indices[0])

    relevant_chunks = []
    for i, idx in enumerate(retrieved_indices):
        if idx != -1 and idx < len(chunks):
            relevant_chunks.append(chunks[idx])
            print(f"    {i+1}. Chunk {idx} (similarity: {distances[0][list(indices[0]).index(idx)]:.3f})")

    combined = '\n\n'.join(relevant_chunks)

    # Stage 6: Token budget check
    retrieved_tokens = len(tokenizer.encode(combined))
    print(f"  Retrieved context: {retrieved_tokens} tokens")

    if retrieved_tokens > 3500:
        context_tokens = tokenizer.encode(combined)[:3500]
        combined = tokenizer.decode(context_tokens, skip_special_tokens=True)
        print(f"  Truncated to 3500 tokens")

    # Profile BEFORE inference
    gc.collect()
    ram_before_inference = psutil.virtual_memory().used / 1e9

    # Stage 7: Generate answer
    prompt = f"""Context:
{combined}

Question: {query}

Answer:"""

    response = ask(prompt)

    # Profile AFTER inference
    gc.collect()
    ram_after_inference = psutil.virtual_memory().used / 1e9
    print(f"  RAM during inference: {ram_after_inference:.2f} GB")

    if "Answer:" in response:
        answer = response.split("Answer:")[-1].strip()
    else:
        answer = response.strip()

    return answer

# Test with RAM tracking
print(f"Testing: {test_queries[0]}")
print(f"RAM at start of test: {psutil.virtual_memory().used / 1e9:.2f} GB\n")

start = time.time()
answer3 = baseline_dolphinmind(long_context, test_queries[0])
time3 = time.time() - start

print(f"\nAnswer: {answer3}")
print(f"Time: {time3:.2f}s")

# Final RAM measurement
gc.collect()
print(f"RAM at end of test: {psutil.virtual_memory().used / 1e9:.2f} GB\n")

# Calculate DolphinMind-specific overhead
print("="*60)
print("DOLPHINMIND RAM BREAKDOWN")
print("="*60)
print(f"Base system (model loaded): {ram_before_embedding:.2f} GB")
print(f"+ Embedding model: {ram_after_embedding - ram_before_embedding:.2f} GB")
print(f"+ FAISS index (during query): ~0.3-0.5 GB (dynamic)")
print(f"= Total DolphinMind overhead: ~{(ram_after_embedding - ram_before_embedding) + 0.4:.2f} GB")
print(f"= Peak RAM during operation: ~{ram_after_embedding + 0.5:.2f} GB")
print("="*60)

=== BASELINE 3: DolphinMind RAG (Overlapping Chunks + Semantic Retrieval) ===
Loading embedding model...
RAM before embedding model: 5.28 GB
RAM after embedding model: 5.34 GB
Embedding model cost: 0.06 GB
Loaded

Testing: Who is Sir Francis Varney?
RAM at start of test: 5.34 GB

  Context too long (453229 tokens), using DolphinMind pipeline
  Created 862 overlapping chunks (size=512, overlap=128)
  RAM after FAISS indexing: 5.78 GB
  FAISS index cost: 0.30 GB
  Retrieved top 5 chunks
    1. Chunk 514 (similarity: 0.604)
    2. Chunk 540 (similarity: 0.632)
    3. Chunk 653 (similarity: 0.615)
    4. Chunk 682 (similarity: 0.626)
    5. Chunk 731 (similarity: 0.599)
  Retrieved context: 3879 tokens
  Truncated to 3500 tokens
  RAM during inference: 5.59 GB

Answer: Sir Francis Varney is described as a vampyre, a supernatural being that can be killed by a stake through the heart. He is also referred to as a prisoner, a thief, and a murderer, though he denies these accusations. He is a m

In [11]:
print("\n=== EVALUATING ALL BASELINES ===\n")

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

results = []

for method_name, method_func in [
    ("Truncated", baseline_truncated),
    ("Naive Chunking", baseline_naive_chunking),
    ("DolphinMind RAG", baseline_dolphinmind)  # Changed from dolphinmind_rag
]:
    print(f"Evaluating {method_name}...")

    predictions = []
    times = []

    for i, (query, ref) in enumerate(zip(test_queries, reference_answers)):
        print(f"  Query {i+1}/{len(test_queries)}: {query[:50]}...")
        start = time.time()
        pred = method_func(long_context, query)
        times.append(time.time() - start)
        predictions.append(pred)

    rouge_scores = [
        scorer.score(ref, pred)['rougeL'].fmeasure
        for pred, ref in zip(predictions, reference_answers)
    ]
    avg_rouge = np.mean(rouge_scores)
    avg_time = np.mean(times)

    results.append({
        'method': method_name,
        'rouge_l': avg_rouge,
        'avg_time': avg_time
    })

    print(f"  ROUGE-L: {avg_rouge:.3f}")
    print(f"  Avg Time: {avg_time:.2f}s\n")

print("Evaluation complete")


=== EVALUATING ALL BASELINES ===

Evaluating Truncated...
  Query 1/3: Who is Sir Francis Varney?...
  Query 2/3: What happens to Flora Bannerworth?...
  Query 3/3: Who is Admiral Bell and what role does he play?...
  ROUGE-L: 0.146
  Avg Time: 7.79s

Evaluating Naive Chunking...
  Query 1/3: Who is Sir Francis Varney?...
Created 659 chunks
  Query 2/3: What happens to Flora Bannerworth?...
Created 659 chunks
  Query 3/3: Who is Admiral Bell and what role does he play?...
Created 659 chunks
  ROUGE-L: 0.135
  Avg Time: 6.31s

Evaluating DolphinMind RAG...
  Query 1/3: Who is Sir Francis Varney?...
  Context too long (453229 tokens), using DolphinMind pipeline
  Created 862 overlapping chunks (size=512, overlap=128)
  RAM after FAISS indexing: 7.21 GB
  FAISS index cost: 0.02 GB
  Retrieved top 5 chunks
    1. Chunk 514 (similarity: 0.604)
    2. Chunk 540 (similarity: 0.632)
    3. Chunk 653 (similarity: 0.615)
    4. Chunk 682 (similarity: 0.626)
    5. Chunk 731 (similarity: 0.599)


In [12]:
print("\n=== FINAL RESULTS ===\n")

# Simple text table
print(f"{'Method':<25} {'ROUGE-L':>10} {'Avg Time (s)':>15}")
print("-" * 52)

for r in results:
    print(f"{r['method']:<25} {r['rouge_l']:>10.3f} {r['avg_time']:>15.2f}")

print("-" * 52)

# Find best
best = max(results, key=lambda x: x['rouge_l'])
fastest = min(results, key=lambda x: x['avg_time'])

print(f"\nBest ROUGE-L: {best['method']} ({best['rouge_l']:.3f})")
print(f"Fastest: {fastest['method']} ({fastest['avg_time']:.2f}s)")

profile()


=== FINAL RESULTS ===

Method                       ROUGE-L    Avg Time (s)
----------------------------------------------------
Truncated                      0.146            7.79
Naive Chunking                 0.135            6.31
DolphinMind RAG                0.185           22.54
----------------------------------------------------

Best ROUGE-L: DolphinMind RAG (0.185)
Fastest: Naive Chunking (6.31s)


RAM Used: 7.15 GB

In [13]:
import json

results_dict = {
    'experiment': 'baseline_comparison',
    'model': model_name,
    'context_length': len(long_context.split()),
    'num_queries': len(test_queries),
    'results': results
}

with open('baseline_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

print("\nResults saved to baseline_results.json")


Results saved to baseline_results.json


In [14]:
print("\n===SUMMARY ===\n")
print(f"tested 3 baseline methods")
print(f"context length: {len(long_context.split()):,} words")
print(f"evaluated on {len(test_queries)} queries")
print(f"memory usage: {psutil.virtual_memory().used / 1e9:.2f} GB")


===SUMMARY ===

tested 3 baseline methods
context length: 329,160 words
evaluated on 3 queries
memory usage: 7.14 GB
